[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traceopt-ai/traceml/blob/main/notebooks/huggingface_dataloading_bottleneck.ipynb)

# Find a data-loading bottleneck with Hugging Face Trainer

A training job can keep making progress while its GPU repeatedly waits for the next batch. This notebook makes that wait visible, then fixes it without rewriting a training loop.

You will train the same Hugging Face ResNet-50 job twice on the real 320px Imagenette dataset. The only difference is three `TrainingArguments` data-loader settings: workers, pinned memory, and persistent workers. For each run, TraceML reports the diagnosis; the final cell compares the saved summaries with `traceml compare`.

The result is intentionally hardware-dependent. More CPU cores can decode images in parallel and keep the GPU fed; a small Colab CPU will still benefit, but may remain input-bound. That is useful information about *your* machine, not a failed demo.

**Before you start:** in Colab, choose **Runtime → Change runtime type → GPU**, then run the cells from top to bottom. This notebook downloads a 326 MiB image archive and a ResNet-50 checkpoint.

## Choose a run mode

`extended` is the default GPU/Imagenette demonstration. `smoke` is a small CPU-only verification mode for contributors and CI: it uses synthetic data, downloads nothing, and checks that TraceML produces a complete input-bound diagnosis.


In [ ]:
import os

# Change the default to "smoke" to run the small CPU-only path manually.
RUN_MODE = os.environ.get("TRACEML_NOTEBOOK_MODE", "extended")
if RUN_MODE not in {"extended", "smoke"}:
    raise ValueError("TRACEML_NOTEBOOK_MODE must be 'extended' or 'smoke'")
print(f"TraceML notebook mode: {RUN_MODE}")

## 1. Check that a GPU is available (extended mode only)


In [ ]:
import torch

if RUN_MODE == "extended":
    !nvidia-smi -L
    print("CUDA available:", torch.cuda.is_available())
    assert (
        torch.cuda.is_available()
    ), "No GPU found. In Colab: Runtime → Change runtime type → GPU, then rerun."
else:
    print("Smoke mode uses CPU; no GPU is required.")

## 2. Install the notebook dependencies

TraceML adds `TraceMLTrainerCallback` to the standard Hugging Face `Trainer`. The notebook uses ordinary `TrainingArguments` and `Trainer`; there is no custom training loop to learn.

Colab supplies a CUDA-matched PyTorch and torchvision build, so this cell installs TraceML and the Hugging Face integration dependencies without replacing them. Outside Colab, install the Torch extra shown in the comment below.

In [ ]:
import os

if os.environ.get("TRACEML_NOTEBOOK_SKIP_INSTALL") == "1":
    print("Notebook smoke runner provided TraceML dependencies.")
else:
    %pip install -q "traceml-ai[hf]"
# Outside Colab or in a fresh CPU environment: %pip install -q "traceml-ai[torch,hf]"

## 3. Download a compact dataset where image decoding still matters (extended mode only)

The extended GPU demonstration uses the official 320px Imagenette variant. Smoke mode skips this download and creates its data and model in memory.


In [ ]:
import os

if RUN_MODE == "extended":
    if not os.path.isdir("imagenette2-320/train"):
        !wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz
        !tar -xzf imagenette2-320.tgz

    print("CPU cores available:", os.cpu_count())
    print(
        "Training images:",
        sum(len(files) for _, _, files in os.walk("imagenette2-320/train")),
    )
else:
    print("Smoke mode uses a small in-memory synthetic dataset and model.")

## 4. The complete Trainer script

This is ordinary Hugging Face training: an AutoModel, `TrainingArguments`, and a `Trainer`-style call to `train()`. The TraceML-specific pieces are deliberately small:

1. Call `traceml_hf.init()` once so TraceML can measure internally-created DataLoader fetches.
2. Add `traceml_hf.TraceMLTrainerCallback()` to the standard `Trainer` callbacks.

The `--profile` flag is the experiment switch. Model, images, batch size, augmentation, and number of steps stay fixed.

In [ ]:
%%writefile hf_train.py
import argparse
import os
import time

import torch
from torch.utils.data import Dataset
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Normalize, RandomHorizontalFlip, RandomResizedCrop, ToTensor
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    DefaultDataCollator,
    ResNetConfig,
    ResNetForImageClassification,
    Trainer,
    TrainingArguments,
)

from traceml_ai.integrations import huggingface as traceml_hf

MODEL_NAME = "microsoft/resnet-50"


class ImagenetteForTrainer(Dataset):
    # Return the field names expected by AutoModelForImageClassification.

    def __init__(self, root, transform):
        self.images = ImageFolder(root, transform=transform)
        self.classes = self.images.classes

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        pixel_values, label = self.images[index]
        return {"pixel_values": pixel_values, "labels": label}


class SlowSyntheticImages(Dataset):
    # Small CPU-only dataset with deliberate fetch latency for smoke mode.

    def __init__(self, samples=32, delay_s=0.05):
        self.pixel_values = torch.zeros(samples, 3, 32, 32)
        self.labels = torch.arange(samples) % 2
        self.delay_s = delay_s

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        time.sleep(self.delay_s)
        return {
            "pixel_values": self.pixel_values[index],
            "labels": self.labels[index],
        }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--profile", choices=["baseline", "optimized"], required=True)
    parser.add_argument("--data-dir", default="imagenette2-320")
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--max-steps", type=int, default=200)
    parser.add_argument("--smoke", action="store_true",
                        help="Run the small CPU-only synthetic verification path.")
    args = parser.parse_args()

    torch.manual_seed(42)
    optimized = args.profile == "optimized"
    # The only experimental change in the extended demo: DataLoader settings.
    num_workers = 0 if args.smoke else (min(4, os.cpu_count() or 2) if optimized else 0)

    if args.smoke:
        # The short, intentional fetch delay makes the CPU smoke diagnosis
        # stable while exercising the standard Trainer callback integration.
        train_dataset = SlowSyntheticImages()
        model = ResNetForImageClassification(ResNetConfig(
            num_labels=2,
            id2label={0: "zero", 1: "one"},
            label2id={"zero": 0, "one": 1},
            embedding_size=8,
            hidden_sizes=[8, 16, 32, 64],
            depths=[1, 1, 1, 1],
            layer_type="basic",
        ))
    else:
        image_processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
        crop_size = image_processor.size.get("height", image_processor.size.get("shortest_edge", 224))
        transform = Compose([
            RandomResizedCrop(crop_size),
            RandomHorizontalFlip(),
            ToTensor(),
            Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
        ])
        train_dataset = ImagenetteForTrainer(
            os.path.join(args.data_dir, "train"), transform=transform
        )
        id2label = {index: name for index, name in enumerate(train_dataset.classes)}
        label2id = {name: index for index, name in id2label.items()}
        model = AutoModelForImageClassification.from_pretrained(
            MODEL_NAME,
            num_labels=len(train_dataset.classes),
            id2label=id2label,
            label2id=label2id,
            ignore_mismatched_sizes=True,
        )

    training_args = TrainingArguments(
        output_dir=f"outputs/{args.profile}",
        per_device_train_batch_size=args.batch_size,
        max_steps=args.max_steps,
        learning_rate=1e-4,
        save_strategy="no",
        report_to="none",
        disable_tqdm=True,
        remove_unused_columns=False,
        dataloader_num_workers=num_workers,
        dataloader_pin_memory=optimized and not args.smoke,
        dataloader_persistent_workers=optimized and num_workers > 0,
        use_cpu=args.smoke,
    )

    print(
        f"[demo] profile={args.profile} smoke={args.smoke} workers={num_workers} "
        f"pin_memory={optimized and not args.smoke} "
        f"persistent_workers={optimized and num_workers > 0} "
        f"batch_size={args.batch_size} max_steps={args.max_steps}",
        flush=True,
    )

    traceml_hf.init()  # Required once: enables DataLoader fetch and step timing.
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=DefaultDataCollator(),
        callbacks=[traceml_hf.TraceMLTrainerCallback()],
    )
    trainer.train()


if __name__ == "__main__":
    main()


## 5. Run the baseline

Extended mode runs the original GPU/Imagenette baseline. Smoke mode runs eight CPU-only synthetic steps with deliberate input delay, using the standard `Trainer` and `TraceMLTrainerCallback`; it should produce a complete `INPUT-BOUND` diagnosis.


In [ ]:
if RUN_MODE == "smoke":
    !traceml run --mode summary --logs-dir logs --run-name hf_smoke hf_train.py --args --profile baseline --smoke --max-steps 8 --batch-size 4
else:
    !traceml run --mode summary --logs-dir logs --run-name hf_baseline hf_train.py --args --profile baseline --data-dir imagenette2-320 --max-steps 200 --batch-size 32

## 6. Run the optimized loader (extended mode only)

Same images, model, batch size, and steps. This profile lets up to four CPU workers decode ahead, pins batches for faster transfer, and keeps workers alive after the loader is created. If your CPU can hide the decode work, the verdict can flip to `COMPUTE-BOUND`.


In [ ]:
if RUN_MODE == "extended":
    !traceml run --mode summary --logs-dir logs --run-name hf_optimized hf_train.py --args --profile optimized --data-dir imagenette2-320 --max-steps 200 --batch-size 32
else:
    print(
        "Smoke mode runs one intentional input-bound case, not a comparison."
    )

## 7. Compare the two runs

TraceML writes a portable `final_summary.json` for each run. `traceml compare` prints a compact comparison and writes JSON and text artifacts for the baseline-versus-optimized result.

In [ ]:
if RUN_MODE == "smoke":
    import json
    from pathlib import Path

    summary_dir = Path("logs/hf_smoke")
    summary_json = summary_dir / "final_summary.json"
    summary_text = summary_dir / "final_summary.txt"
    assert summary_json.is_file(), f"Missing {summary_json}"
    assert summary_text.is_file(), f"Missing {summary_text}"
    summary = json.loads(summary_json.read_text())
    diagnosis = summary["primary_diagnosis"]["status"]
    assert (
        diagnosis == "INPUT-BOUND"
    ), f"Expected the intentional input-bound smoke diagnosis, got {diagnosis!r}"
    print(f"Smoke check passed: {diagnosis}; artifacts are in {summary_dir}")
else:
    !traceml compare logs/hf_baseline/final_summary.json logs/hf_optimized/final_summary.json --output=logs/hf_baseline_vs_optimized

## Use this in your own Trainer

The experiment is a pattern, not a special ResNet trick. In an existing Hugging Face script, call `traceml_hf.init()` once and add `traceml_hf.TraceMLTrainerCallback()` to the standard `Trainer` callbacks. Then run it through `traceml run --mode summary ...`.

If TraceML says `INPUT-BOUND`, start by testing `dataloader_num_workers`, `dataloader_pin_memory`, and `dataloader_persistent_workers` one change at a time. Keep the change only when the measured wall time and input wait improve on the hardware that will actually run your job.